# Trabalho Prático Final — IA Generativa (UFMG 2026/2)
## Latent Diffusion / Rectified Flow no Dataset EMNIST (36 Classes) com Classifier-Free Guidance

**Professor:** Vinicius M. Gonçalves  
**Dataset:** EMNIST (`36 classes`: 10 dígitos `0-9` e 26 letras `A-Z`)  

### Pipeline Implementado
1. **Variational Autoencoder (VAE — Parte 5):**
   - Encoder probabilístico parametrizado por $\mu_E(x)$ e $\sigma_E(x)$: $Z(x) = \mu_E(x) + \sigma_E(x) W_E$, $W_E \sim \mathcal{N}(0, I)$.
   - Decoder $\mu_D(z)$ reconstruindo imagens $28 \times 28$.
   - Função objetivo: minimização conjunta do erro de reconstrução $R(x; \mu_D, \mu_E, \sigma_E)$ e do erro de distribuição latente $L(x; \mu_D, \mu_E, \sigma_E)$.
2. **Rectified Flow / Flow Matching no Espaço Latente ($\Phi$ — Parte 6):**
   - Interpolação retificada no espaço latente (parametrização *forward* $t \in [0, 1]$): $z(t) = (1 - t) z(0) + t z(1)$, com $z(0) \sim \mathcal{N}(0, I)$ e $z(1) = \mu_E(x) + \sigma_E(x) W$.
   - Treinamento do campo de velocidade $\Phi(z(t), t, c)$ minimizando $\|\Phi(z(t), t, c) - (z(1) - z(0))\|^2$.
3. **Condicionamento nas 36 Classes & Classifier-Free Guidance (Bônus +5 pts — Parte 7):**
   - Treinamento conjunto condicional e incondicional (descarte aleatório do rótulo $c \to \varnothing$ com probabilidade $p_{\text{uncond}} = 0.15$).
   - Amostragem via **Euler** e **Heun** integrando $z(0) \to z(1)$ com velocidade guiada:
     $$v_{\text{CFG}}(z, t, c) = \Phi(z, t, \varnothing) + w \cdot \big(\Phi(z, t, c) - \Phi(z, t, \varnothing)\big)$$
   - Decodificação final: $x = \mu_D(z(1))$.

In [ ]:
# 1. Configuração do ambiente no Google Colab (clona o repositório caso necessário)
import os
import sys

REPO_URL = "https://github.com/Gui-Domenico-UFMG/ia-generativa-ufmg-latent-flow.git"
REPO_DIR = "ia-generativa-ufmg-latent-flow"

if not os.path.exists("src") and not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}
if os.path.exists(REPO_DIR) and REPO_DIR not in sys.path:
    os.chdir(REPO_DIR)

import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo em uso: {device}")

In [ ]:
# 2. Carregamento do Dataset EMNIST (36 classes: '0'-'9' e 'A'-'Z')
from src.dataset import get_dataloaders, CLASS_NAMES, NUM_CLASSES, index_to_char

BATCH_SIZE = 256
train_loader, test_loader = get_dataloaders(data_dir="./data", batch_size=BATCH_SIZE)

print(f"Total de classes: {NUM_CLASSES} -> {CLASS_NAMES}")
print(f"Batches de treino: {len(train_loader)} | Batches de teste: {len(test_loader)}")

# Visualizar algumas amostras reais do dataset
sample_x, sample_y = next(iter(test_loader))
fig, axes = plt.subplots(2, 8, figsize=(11, 3))
for i, ax in enumerate(axes.flat):
    ax.imshow(sample_x[i, 0], cmap="gray")
    ax.set_title(f"Classe: {index_to_char(int(sample_y[i]))}", fontsize=9)
    ax.axis("off")
plt.suptitle("Amostras Reais do EMNIST (36 Classes)", fontsize=12)
plt.tight_layout()
plt.show()

## Etapa 1: Treinamento do Variational Autoencoder (VAE)
Seguindo a **Parte 5**, otimizamos as redes $\mu_E(x, \theta)$, $\sigma_E(x, \theta)$ e $\mu_D(z, \delta)$ minimizando:
$$\min_{\mu_D, \mu_E, \sigma_E} \sum_{i=1}^N \Big\{ R(x_i; \mu_D, \mu_E, \sigma_E) + \beta L(x_i; \mu_D, \mu_E, \sigma_E) \Big\}$$

In [ ]:
from src.vae import VAE
from src.vis import plot_vae_reconstructions

LATENT_DIM = 16
SCALAR_SIGMA = False  # Altere para True se quiser sigma_E escalar exato da Eq. (29)/(47)
BETA = 0.25
VAE_EPOCHS = 15
LR = 1e-3

vae = VAE(latent_dim=LATENT_DIM, scalar_sigma=SCALAR_SIGMA).to(device)
opt_vae = torch.optim.AdamW(vae.parameters(), lr=LR, weight_decay=1e-4)

vae_history = []
for epoch in range(1, VAE_EPOCHS + 1):
    vae.train()
    ep_loss, ep_r, ep_l = 0.0, 0.0, 0.0
    for x, _ in tqdm(train_loader, desc=f"VAE Época {epoch}/{VAE_EPOCHS}", leave=False):
        x = x.to(device)
        opt_vae.zero_grad()
        losses = vae.compute_loss(x, beta=BETA)
        losses["loss"].backward()
        opt_vae.step()

        ep_loss += losses["loss"].item()
        ep_r += losses["recon_error"].item()
        ep_l += losses["latent_error"].item()

    n_b = len(train_loader)
    vae_history.append((ep_loss / n_b, ep_r / n_b, ep_l / n_b))
    print(f"[VAE {epoch:02d}/{VAE_EPOCHS:02d}] Loss: {ep_loss/n_b:.3f} | R(x): {ep_r/n_b:.3f} | L(x): {ep_l/n_b:.3f}")

os.makedirs("outputs", exist_ok=True)
torch.save(vae.state_dict(), "outputs/vae_emnist36.pt")
plot_vae_reconstructions(vae, sample_x, sample_y, num_display=10, save_path="outputs/vae_reconstructions.png")

## Etapa 2: Treinamento do Campo de Velocidade ($\Phi$) no Espaço Latente com Condicionamento (CFG)
Seguindo o **Slide 70 da Parte 6** e a **Parte 7**:
1. Codificamos o dataset de treino para o espaço latente usando o Encoder $\mu_E(x), \sigma_E(x)$.
2. Treinamos o modelo de velocidade $\Phi(z, t, c)$ para transportar amostras de $z(0) \sim \mathcal{N}(0, I)$ até $z(1) \sim \mathcal{N}(\mu_E(x), \sigma_E(x)^2 I)$ ao longo de retas $z(t) = (1-t)z(0) + t z(1)$.

In [ ]:
from src.flow import LatentVelocityField
from train import encode_dataset_to_latent

latent_loader = encode_dataset_to_latent(vae, train_loader, device)

FLOW_EPOCHS = 35
flow = LatentVelocityField(
    latent_dim=LATENT_DIM,
    hidden_dim=256,
    cond_dim=128,
    num_blocks=4,
    num_classes=NUM_CLASSES,
).to(device)

opt_flow = torch.optim.AdamW(flow.parameters(), lr=1e-3, weight_decay=1e-4)
sched_flow = torch.optim.lr_scheduler.CosineAnnealingLR(opt_flow, T_max=FLOW_EPOCHS)

flow_losses = []
for epoch in range(1, FLOW_EPOCHS + 1):
    flow.train()
    ep_flow_loss = 0.0
    for mu_e, sigma_e, y in latent_loader:
        mu_e, sigma_e, y = mu_e.to(device), sigma_e.to(device), y.to(device)
        z_1 = mu_e + sigma_e * torch.randn_like(mu_e)

        opt_flow.zero_grad()
        loss = flow.compute_flow_matching_loss(z_1=z_1, labels=y, p_uncond=0.15)
        loss.backward()
        opt_flow.step()
        ep_flow_loss += loss.item()

    sched_flow.step()
    avg_fl = ep_flow_loss / len(latent_loader)
    flow_losses.append(avg_fl)
    if epoch % 5 == 0 or epoch == 1 or epoch == FLOW_EPOCHS:
        print(f"[Flow {epoch:02d}/{FLOW_EPOCHS:02d}] Flow Matching MSE Loss: {avg_fl:.4f}")

torch.save(flow.state_dict(), "outputs/flow_emnist36.pt")

## Etapa 3: Geração de Novas Amostras Controladas por Classe ($z(0) \to z(1) \to x = \mu_D(z(1))$)
- Avaliamos a geração condicional nas **36 classes** (`0-9` e `A-Z`) usando **Classifier-Free Guidance (CFG)** e o sampler de **Heun** (2ª ordem) e **Euler** (1ª ordem).
- Visualizamos também a trajetória contínua no espaço latente decodificada passo a passo de $t=0$ a $t=1$.

In [ ]:
from src.vis import plot_all_36_classes, plot_flow_trajectories

# Grade 6x6 com todas as 36 classes geradas via Heun + CFG
plot_all_36_classes(
    vae,
    flow,
    device=device,
    num_steps=50,
    guidance_scale=2.5,
    sampler="heun",
    save_path="outputs/generated_36_classes.png",
)

# Trajetória do fluxo latente z(0) -> z(1) decodificada em imagens
plot_flow_trajectories(
    vae,
    flow,
    device=device,
    target_chars=["0", "3", "7", "9", "A", "B", "G", "M"],
    num_steps=50,
    guidance_scale=2.5,
    sampler="heun",
    save_path="outputs/flow_trajectories.png",
)